# Lesson 04 — Aspect Ratio, Extent, and Solidity

## Why
These three simple ratios let you classify shapes without any ML.
Circle vs rectangle vs star — pure geometry.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def shape_features(c):
    area      = cv2.contourArea(c)
    perimeter = cv2.arcLength(c, True)
    x,y,w,h   = cv2.boundingRect(c)
    hull_area = cv2.contourArea(cv2.convexHull(c))
    rect_area = w * h

    aspect_ratio = w / (h + 1e-6)
    extent       = area / (rect_area + 1e-6)    # how much of bbox is filled
    solidity     = area / (hull_area  + 1e-6)   # how convex (1=fully convex)
    circularity  = 4 * np.pi * area / (perimeter**2 + 1e-6)  # 1=perfect circle

    return aspect_ratio, extent, solidity, circularity

canvas = np.zeros((300, 700), dtype=np.uint8)
cv2.rectangle(canvas, (30,  80), (180, 220), 255, -1)   # square
cv2.ellipse(canvas,   (310,150), (120,60),  0,0,360, 255,-1)  # ellipse
cv2.circle(canvas,    (560,150), 100, 255, -1)           # circle

# Add a star-like shape
star = np.array([[480,50],[500,120],[570,120],[515,165],
                 [535,240],[480,195],[425,240],[445,165],[390,120],[460,120]], dtype=np.int32)
cv2.fillPoly(canvas, [star], 0)  # erase from circle to make crescent — skip this for real star

contours, _ = cv2.findContours(canvas, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
vis = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
shapes = ['Square', 'Ellipse', 'Circle']

for c, name in zip(sorted(contours, key=lambda c: cv2.boundingRect(c)[0]), shapes):
    ar, ext, sol, circ = shape_features(c)
    x,y,w,h = cv2.boundingRect(c)
    cv2.putText(vis, f'{name}', (x, y-30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,255), 2)
    cv2.putText(vis, f'sol={sol:.2f} circ={circ:.2f}', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,0), 1)
    cv2.drawContours(vis, [c], -1, (0,255,0), 2)
    print(f"{name}: AR={ar:.2f} Extent={ext:.2f} Solidity={sol:.2f} Circularity={circ:.2f}")

plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## Key Takeaway
Circularity ≈ 1 → circle. Solidity < 0.8 → concave shape (star, hand, crescent).
Aspect ratio >> 1 → wide/horizontal. Aspect ratio << 1 → tall/vertical.
These 4 features classify most simple shapes without any deep learning.